In [ ]:
# save_faces.py
import cv2
import os

cam = cv2.VideoCapture(0)
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

face_id = 1 
count = 0

os.makedirs("dataset", exist_ok=True)

while True:
    ret, img = cam.read()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        count += 1
        face_img = gray[y:y+h, x:x+w]
        cv2.imwrite(f"dataset/admin.{face_id}.{count}.jpg", face_img)
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)

    cv2.imshow('Capturing Faces', img)
    if cv2.waitKey(100) & 0xFF == ord('q') or count >= 30:
        break

cam.release()
cv2.destroyAllWindows()
print(" Face samples collected.")

 Face samples collected.


In [20]:
# train_model.py
import cv2
import numpy as np
from PIL import Image
import os

path = 'dataset'
recognizer = cv2.face.LBPHFaceRecognizer_create()
detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def get_images_and_labels(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    face_samples = []
    ids = []

    for imagePath in image_paths:
        pil_img = Image.open(imagePath).convert('L')  # grayscale
        img_numpy = np.array(pil_img, 'uint8')
        id = int(os.path.split(imagePath)[-1].split(".")[1])
        faces = detector.detectMultiScale(img_numpy)

        for (x, y, w, h) in faces:
            face_samples.append(img_numpy[y:y+h, x:x+w])
            ids.append(id)

    return face_samples, ids

faces, ids = get_images_and_labels(path)
recognizer.train(faces, np.array(ids))
recognizer.write('trainer.yml')

print(" Model trained and saved as 'trainer.yml'")


 Model trained and saved as 'trainer.yml'


In [ ]:
import cv2

recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("trainer.yml")

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cam = cv2.VideoCapture(0)
if not cam.isOpened():
    print(" Error: Could not open webcam.")
    exit()

print("📸 Starting face recognition... Press 'q' to quit.")
unlocked = False  # flag

while True:
    ret, img = cam.read() 
    if not ret or img is None:
        print(" Warning: Failed to grab frame.")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.2, 5)

    for (x, y, w, h) in faces:
        id_, confidence = recognizer.predict(gray[y:y+h, x:x+w])
        if confidence < 50:
            print(" Admin recognized! Access granted.")
            unlocked = True
            break
        else:
            print(f" Unknown face (confidence: {round(confidence, 2)})")
    
    # Break outer loop if access granted
    if unlocked:
        break

    cv2.imshow('Face Unlock', img)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()



📸 Starting face recognition... Press 'q' to quit.
 Admin recognized! Access granted.
